In [2]:
import pandas as pd
import joblib

train = pd.read_parquet("data/processed/notebook5_train_features.parquet")
val = pd.read_parquet("data/processed/notebook5_val_features.parquet")
test = pd.read_parquet("data/processed/notebook5_test_features.parquet")

feature_names = joblib.load("data/models/notebook5_feature_names.pkl")

print("train:", train.shape)
print("val:", val.shape)
print("test:", test.shape)
print("number of features:", len(feature_names))

train: (67529, 32)
val: (14470, 32)
test: (14471, 32)
number of features: 31


In [3]:
X_train = train[feature_names]
y_train = train["label"]

X_val = val[feature_names]
y_val = val["label"]

X_test = test[feature_names]
y_test = test["label"]

print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_val:", X_val.shape, "| y_val:", y_val.shape)
print("X_test:", X_test.shape, "| y_test:", y_test.shape)

X_train: (67529, 31) | y_train: (67529,)
X_val: (14470, 31) | y_val: (14470,)
X_test: (14471, 31) | y_test: (14471,)


------------------------------------------------------------------------------------------------------------------------------------------

# **Baseline**

In [4]:
# كود Baseline (DummyClassifier)
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

y_val_pred_baseline = baseline.predict(X_val)

print(classification_report(y_val, y_val_pred_baseline))

              precision    recall  f1-score   support

        Late       0.00      0.00      0.00       773
     On-time       0.95      1.00      0.97     13697

    accuracy                           0.95     14470
   macro avg       0.47      0.50      0.49     14470
weighted avg       0.90      0.95      0.92     14470



/home/lolia/olist-eda/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/lolia/olist-eda/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/lolia/olist-eda/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", res

### Baseline (DummyClassifier - most_frequent) على val

- **Accuracy:** 95% (رقم مضلل تمامًا)

- **Late:** Precision=0, Recall=0, F1=0 (الموديل ما تنبأ بـ Late إطلاقًا)

- **On-time:** Precision=0.95, Recall=1.00, F1=0.97

- هذا يثبت عمليًا أن الـ accuracy وحدها مقياس غير كافٍ لمشكلتنا (imbalanced classification) — أي موديل حقيقي لازم يحقق F1-score أعلى من صفر لفئة Late ليكون ذا قيمة فعلية.

- هذا هو المعيار المرجعي (benchmark) الذي يجب على الموديل الحقيقي التغلب عليه.

# **LogisticRegression**

In [5]:
# كود تدريب LR الأساسي
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(class_weight="balanced", max_iter=1000)
log_reg.fit(X_train, y_train)

y_val_pred_lr = log_reg.predict(X_val)

print(classification_report(y_val, y_val_pred_lr))

              precision    recall  f1-score   support

        Late       0.09      0.46      0.15       773
     On-time       0.96      0.74      0.84     13697

    accuracy                           0.73     14470
   macro avg       0.53      0.60      0.49     14470
weighted avg       0.91      0.73      0.80     14470



### Logistic Regression (`class_weight="balanced"`) على val

- **Accuracy:** 73% (أقل من الـ baseline 95%، لكن هذا متوقع ومقبول)

- **Late:** Precision=0.09, Recall=0.46, F1=0.15

- **On-time:** Precision=0.96, Recall=0.74, F1=0.84

### مقارنة مع Baseline

- Late F1-score تحسن من 0.00 إلى 0.15 — تحسن حقيقي وذو معنى.

- Late Recall تحسن من 0.00 إلى 0.46 — الموديل يمسك الآن ~46% من حالات التأخير الحقيقية، مقابل صفر بالـ baseline.

- الانخفاض في الـ accuracy (95%→73%) هو نتيجة طبيعية لـ `class_weight="balanced"`، الذي يجعل الموديل "يجازف" أكثر بالتنبؤ بـ Late، فيقل الـ accuracy الكلي (لأن معظم البيانات On-time) لكن يتحسن الأداء الفعلي على الفئة النادرة والمهمة.

- Trade-off واضح: Precision منخفض جدًا (0.09) يعني الكثير من false positives (طلبيات On-time صُنّفت خطأً كـ Late).

- الخلاصة: تحسن حقيقي وملموس مقارنة بالـ baseline، لكن الأداء لسا بعيد عن المثالي — يستدعي تجربة موديل ثاني (Random Forest) ومقارنة الأداء.

# **RandomForest**

In [6]:
# كود RF default
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)

y_val_pred_rf = rf.predict(X_val)

print(classification_report(y_val, y_val_pred_rf))

              precision    recall  f1-score   support

        Late       0.08      0.04      0.05       773
     On-time       0.95      0.98      0.96     13697

    accuracy                           0.93     14470
   macro avg       0.51      0.51      0.51     14470
weighted avg       0.90      0.93      0.91     14470



### Random Forest (`class_weight="balanced"`, إعدادات افتراضية) على val

- **Accuracy:** 93% (أعلى من LR لكن مضلل)

- **Late:** Precision=0.08, Recall=0.04, F1=0.05

- **On-time:** Precision=0.95, Recall=0.98, F1=0.96

### مقارنة مع Logistic Regression

- Late F1-score أسوأ بكثير (0.05 مقابل 0.15) رغم Accuracy أعلى — نفس فخ الـ baseline: accuracy عالي بسبب تجاهل الفئة النادرة تقريبًا.

- Recall منخفض جدًا (0.04) يعني الموديل يمسك 4% بس من حالات Late الحقيقية — أداء ضعيف جدًا فعليًا رغم الاسم "القوي" للموديل.

### التفسير

Random Forest بإعداداته الافتراضية (`max_depth=None`) يميل للـ overfitting على الفئة الأغلبية، وأقل استجابة لـ `class_weight` مقارنة بـ Logistic Regression.

### القرار

هذا الموديل بإعداداته الحالية غير مناسب — يحتاج tuning فعلي (`max_depth` محدود، `n_estimators`، إلخ) قبل الحكم النهائي عليه، أو الاستمرار بـ Logistic Regression كأفضل مرشح حاليًا.

In [7]:
# كود RF tuned (max_depth=5)
rf_tuned = RandomForestClassifier(class_weight="balanced", max_depth=5, random_state=42)
rf_tuned.fit(X_train, y_train)

y_val_pred_rf_tuned = rf_tuned.predict(X_val)

print(classification_report(y_val, y_val_pred_rf_tuned))

              precision    recall  f1-score   support

        Late       0.08      0.13      0.10       773
     On-time       0.95      0.92      0.94     13697

    accuracy                           0.88     14470
   macro avg       0.52      0.52      0.52     14470
weighted avg       0.90      0.88      0.89     14470



### Random Forest (`max_depth=5`) على val - بعد Tuning

- **Accuracy:** 88% | **Late:** Precision=0.08, Recall=0.13, F1=0.10

- تحسن ملحوظ مقارنة بـ RF الافتراضي (F1: 0.05→0.10، Recall: 0.04→0.13) — تحديد `max_depth` ساعد الموديل يعمم أكثر بدل حفظ التفاصيل.

- لكن لسا أضعف بوضوح من Logistic Regression (F1=0.15).

- الخلاصة: حتى بعد التعديل، Random Forest لم يتفوق على الموديل الأبسط بهذا المشروع — قد يرجع لطبيعة الـ features (معظمها time/categorical بسيطة نسبيًا، دون علاقات غير خطية معقدة كافية لتستفيد منها أشجار القرار).

# **XGBoost**

In [8]:
# كود حساب scale_pos_weight
neg = (y_train == "On-time").sum()
pos = (y_train == "Late").sum()
scale_pos_weight = neg / pos

print("On-time count:", neg)
print("Late count:", pos)
print("scale_pos_weight:", scale_pos_weight)

On-time count: 61433
Late count: 6096
scale_pos_weight: 10.07759186351706


In [9]:
# كود تدريب XGBoost
from xgboost import XGBClassifier

xgb = XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=42, eval_metric="logloss")

y_train_binary = (y_train == "Late").astype(int)
xgb.fit(X_train, y_train_binary)

y_val_pred_xgb = xgb.predict(X_val)
y_val_pred_xgb_labels = pd.Series(y_val_pred_xgb).map({1: "Late", 0: "On-time"})

print(classification_report(y_val, y_val_pred_xgb_labels))

              precision    recall  f1-score   support

        Late       0.08      0.11      0.09       773
     On-time       0.95      0.93      0.94     13697

    accuracy                           0.89     14470
   macro avg       0.51      0.52      0.51     14470
weighted avg       0.90      0.89      0.89     14470



### XGBoost (`scale_pos_weight=10.08`) على val

- **Accuracy:** 89% | **Late:** Precision=0.08, Recall=0.11, F1=0.09

- أضعف من Logistic Regression (F1=0.15)، وقريب من RF المضبوط (F1=0.10).

### ملاحظة مهمة

كل الموديلات المعقدة (tree-based: RF, XGBoost) أدت أضعف من الموديل الخطي البسيط (Logistic Regression) بهذا المشروع.

هذا بناسب مع ملاحظة Notebook 4 أن الارتباطات الخطية بين الـ features والـ label كانت كلها ضعيفة جدًا (0.02-0.05) — يشير إلى أن الإشارة (signal) المتاحة بالبيانات محدودة بطبيعتها، وليست مسألة اختيار موديل فقط.

### القرار

XGBoost بإعداداته الأساسية غير كافٍ للتفوق على LR. يمكن تجربة tuning إضافي (`max_depth`, `learning_rate`) لكن الفرق المحتمل غير مضمون بناءً على النمط الملحوظ.

# **LogisticRegression2**

In [10]:
# كود Threshold tuning (0.3/0.4/0.5)

# كود GridSearchCV (C, penalty)

# كود تطبيق أفضل C وتقييمه

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import pandas as pd

# تدريب الموديل الأساسي
log_reg = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
log_reg.fit(X_train, y_train)


print(log_reg.classes_)  # ['Late', 'On-time']


y_val_proba_lr = log_reg.predict_proba(X_val)[:, 0]

# تجربة threshold مختلفة
for threshold in [0.3, 0.4, 0.5]:
    y_val_pred_thresh = (y_val_proba_lr >= threshold)
    y_val_pred_thresh_labels = pd.Series(y_val_pred_thresh).map({True: "Late", False: "On-time"})
    
    print(f"--- Threshold = {threshold} ---")
    print(classification_report(y_val, y_val_pred_thresh_labels))

['Late' 'On-time']
--- Threshold = 0.3 ---
              precision    recall  f1-score   support

        Late       0.06      0.92      0.11       773
     On-time       0.97      0.18      0.31     13697

    accuracy                           0.22     14470
   macro avg       0.52      0.55      0.21     14470
weighted avg       0.93      0.22      0.30     14470

--- Threshold = 0.4 ---
              precision    recall  f1-score   support

        Late       0.08      0.57      0.13       773
     On-time       0.96      0.61      0.74     13697

    accuracy                           0.60     14470
   macro avg       0.52      0.59      0.44     14470
weighted avg       0.91      0.60      0.71     14470

--- Threshold = 0.5 ---
              precision    recall  f1-score   support

        Late       0.09      0.46      0.15       773
     On-time       0.96      0.74      0.84     13697

    accuracy                           0.73     14470
   macro avg       0.53      0.60    

In [12]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "C": [0.01, 0.1, 1, 10],
    "penalty": ["l2"]
}

lr_grid = GridSearchCV(
    LogisticRegression(class_weight="balanced", max_iter=500, random_state=42),
    param_grid,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1
)

lr_grid.fit(X_train, y_train)

print("best params :", lr_grid.best_params_)
print("best F1 (macro) during CV:", lr_grid.best_score_)

/home/lolia/olist-eda/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/lolia/olist-eda/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/lolia/olist-eda/.venv/lib/

best params : {'C': 10, 'penalty': 'l2'}
best F1 (macro) during CV: 0.31609580778982277


In [13]:
best_lr = LogisticRegression(C=10, penalty="l2", class_weight="balanced", max_iter=500, random_state=42)
best_lr.fit(X_train, y_train)

y_val_pred_best_lr = best_lr.predict(X_val)

print(classification_report(y_val, y_val_pred_best_lr))

/home/lolia/olist-eda/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


              precision    recall  f1-score   support

        Late       0.09      0.46      0.15       773
     On-time       0.96      0.74      0.84     13697

    accuracy                           0.73     14470
   macro avg       0.53      0.60      0.49     14470
weighted avg       0.91      0.73      0.80     14470



### خلاصة شاملة لكل محاولات تحسين الأداء

- **Threshold tuning** (على LR الافتراضي): جُرّب 0.3/0.4/0.5 → لا تحسن، و0.5 يبقى الأفضل (F1=0.15).

- **C (regularization) tuning** عبر GridSearchCV (C=10, penalty=l2 أفضل حسب f1_macro=0.316): عند التطبيق الفعلي وتقييم F1 لفئة Late تحديدًا، النتيجة طابقت الإعدادات الافتراضية حرفيًا (F1=0.15) — لا تحسن فعلي.

### الخلاصة

Logistic Regression وصل لسقف أدائه الممكن على هذه البيانات. لا الـ threshold ولا الـ regularization استطاعا تحسين F1 لفئة Late أبعد من 0.15.

هذا يتسق مع ملاحظة Notebook 4 أن الإشارة (signal) الخطية بين الـ features وlabel ضعيفة أصلاً (correlations 0.02-0.05)، فأي تعديل على موديل خطي لن يستخرج معلومة إضافية غير موجودة أصلاً.

# **Model Comparison Summary — Validation Set Results**



| # | Model | Late Precision | Late Recall | **Late F1** | Accuracy | القرار |
|---|---|---|---|---|---|---|
| 1 | Baseline (Dummy - most frequent) | 0.00 | 0.00 | **0.00** | 95% | ❌ مرفوض — معيار مرجعي فقط |
| 2 | Logistic Regression (default) | 0.09 | 0.46 | **0.15** | 73% | ✅ **الأفضل — مُختار** |
| 3 | Random Forest (default) | 0.08 | 0.04 | **0.05** | 93% | ❌ مرفوض — أسوأ من LR |
| 4 | Random Forest (max_depth=5) | 0.08 | 0.13 | **0.10** | 88% | ❌ مرفوض — لا يزال أضعف من LR |
| 5 | XGBoost (scale_pos_weight=10.08) | 0.08 | 0.11 | **0.09** | 89% | ❌ مرفوض — أضعف من LR |
| 6 | LR + Threshold=0.3 | 0.06 | 0.92 | **0.11** | 22% | ❌ مرفوض — precision منهار |
| 7 | LR + Threshold=0.4 | 0.08 | 0.57 | **0.13** | 60% | ❌ مرفوض — أضعف من الافتراضي |
| 8 | LR + C=10 (بعد GridSearchCV) | 0.09 | 0.46 | **0.15** | 73% | ➖ مطابق تمامًا للافتراضي |

---

###  القرار النهائي

**Logistic Regression بإعداداته الافتراضية** (`class_weight="balanced"`, `C=1`, `threshold=0.5`) هو **الموديل المختار للتقييم النهائي على test**.

**الأسباب:**
1. حقق أعلى F1-score لفئة Late (0.15) من بين كل الموديلات والإعدادات المجرّبة (8 محاولات مختلفة)
2. تحسن حقيقي وملموس مقارنة بالـ baseline (من F1=0.00 إلى F1=0.15)
3. لا الموديلات الأكثر تعقيدًا (Random Forest, XGBoost) ولا محاولات الـ tuning الإضافية (threshold, regularization) نجحت بتجاوزه
4. هذا يتسق مع ملاحظة Notebook 4: الإشارة الخطية بين الـ features والـ label ضعيفة أصلاً (correlations 0.02-0.05)، فموديل خطي بسيط كافٍ ومناسب لحجم الإشارة المتاح في البيانات

**القيود المعروفة:**
- Precision منخفض جدًا (0.09) — الموديل يعطي إنذارات كاذبة كثيرة (10-11 طلبية يُتوقع تأخرها مقابل كل طلبية متأخرة فعليًا يمسكها بشكل صحيح)
- Recall متوسط (0.46) — يمسك أقل من نصف حالات التأخير الحقيقية
- الأداء العام محدود بسبب ضعف الإشارة في الـ features المتاحة، وليس بسبب اختيار الموديل

# **final_model**

In [14]:
final_model = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
final_model.fit(X_train, y_train)

print("✅")
print("Features:", len(feature_names))

✅
Features: 31


In [15]:
y_test_pred = final_model.predict(X_test)

print("=" * 50)
print("FINAL TEST SET EVALUATION —  run once only")
print("=" * 50)
print(classification_report(y_test, y_test_pred))

FINAL TEST SET EVALUATION —  run once only
              precision    recall  f1-score   support

        Late       0.05      0.12      0.07       957
     On-time       0.93      0.83      0.88     13514

    accuracy                           0.78     14471
   macro avg       0.49      0.47      0.47     14471
weighted avg       0.87      0.78      0.82     14471



### **Final Test Set Evaluation (نُفّذ مرة واحدة فقط، بدون أي تعديل لاحق)**

**Model:** Logistic Regression (`class_weight="balanced"`, default `C=1`)

### نتائج test

- **Accuracy:** 78%
- **Late:** Precision=0.05, Recall=0.12, F1=0.07
- **On-time:** Precision=0.93, Recall=0.83, F1=0.88

### مقارنة مع val (F1=0.15)

- الأداء على test أضعف بشكل ملحوظ (F1 انخفض من 0.15 إلى 0.07، وRecall انخفض من 0.46 إلى 0.12).

### التفسير

يتسق هذا مع ملاحظات Notebook 3 (اختلاف نسبة Late بين train/val/test بسبب time-based split) وNotebook 4 (الاتجاه التصاعدي بنسبة التأخير قرب نهاية فترة train).

النمط الذي تعلمه الموديل من train وضُبط عليه في val قد لا ينطبق بنفس القوة على فترة test (الأحدث زمنيًا) — احتمال وجود "temporal drift" حقيقي في سلوك التأخير بمرور الوقت.

### أهمية هذا الاكتشاف

هذا بالضبط نوع الحقيقة التي يكشفها **time-based split** ويخفيها **random split** — لو استخدمنا تقسيمًا عشوائيًا، كان من المحتمل أن نحصل على تقييم متفائل وغير واقعي لأداء الموديل بالإنتاج الفعلي.

### الخلاصة 

أداء الموديل النهائي محدود جدًا (F1=0.07 على test) — الموديل ليس جاهزًا عمليًا لاستخدام إنتاجي موثوق دون تحسينات جوهرية (features إضافية، بيانات أكثر، أو دراسة أعمق لتغير النمط الزمني). هذا القيد موثق بصدق وليس مخفيًا.

In [16]:
import joblib
import os

os.makedirs("data/models", exist_ok=True)

joblib.dump(final_model, "data/models/notebook6_final_model.pkl")

print(" saved ✅ ")

 saved ✅ 


In [17]:
results_summary = pd.DataFrame([
    {"model": "Baseline (Dummy)", "dataset": "val", "late_precision": 0.00, "late_recall": 0.00, "late_f1": 0.00, "accuracy": 0.95},
    {"model": "Logistic Regression (final)", "dataset": "val", "late_precision": 0.09, "late_recall": 0.46, "late_f1": 0.15, "accuracy": 0.73},
    {"model": "Random Forest (default)", "dataset": "val", "late_precision": 0.08, "late_recall": 0.04, "late_f1": 0.05, "accuracy": 0.93},
    {"model": "Random Forest (max_depth=5)", "dataset": "val", "late_precision": 0.08, "late_recall": 0.13, "late_f1": 0.10, "accuracy": 0.88},
    {"model": "XGBoost", "dataset": "val", "late_precision": 0.08, "late_recall": 0.11, "late_f1": 0.09, "accuracy": 0.89},
    {"model": "Logistic Regression (final)", "dataset": "TEST", "late_precision": 0.05, "late_recall": 0.12, "late_f1": 0.07, "accuracy": 0.78},
])

results_summary.to_csv("data/models/notebook6_results_summary.csv", index=False)
print("saved results summary ✅")
results_summary

saved results summary ✅


,model,dataset,late_precision,late_recall,late_f1,accuracy
0,Baseline (Dummy),val,0.00,0.00,0.00,0.95
1,Logistic Regression (final),val,0.09,0.46,0.15,0.73
2,Random Forest (default),val,0.08,0.04,0.05,0.93
3,Random Forest (max_depth=5),val,0.08,0.13,0.10,0.88
4,XGBoost,val,0.08,0.11,0.09,0.89
5,Logistic Regression (final),TEST,0.05,0.12,0.07,0.78


# **task3**

In [1]:
import mlflow
import mlflow.sklearn
import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("olist-late-delivery")

# Load the already-trained artifacts directly from disk
model = joblib.load("../models/notebook6_final_model.pkl")
feature_names = joblib.load("../models/notebook5_feature_names.pkl")
test_df = pd.read_parquet("../data/processed/notebook5_test_features.parquet")

X_test = test_df[feature_names]
y_test = test_df["label"]

y_pred = model.predict(X_test)
late_idx = list(model.classes_).index("Late")
y_proba_late = model.predict_proba(X_test)[:, late_idx]

with mlflow.start_run(run_name="notebook6_logistic_regression_final"):
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("random_state", 42)
    mlflow.log_param("max_iter", 1000)

    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
    mlflow.log_metric("precision", precision_score(y_test, y_pred, pos_label="Late"))
    mlflow.log_metric("recall", recall_score(y_test, y_pred, pos_label="Late"))
    mlflow.log_metric("f1", f1_score(y_test, y_pred, pos_label="Late"))
    mlflow.log_metric("roc_auc", roc_auc_score((y_test == "Late").astype(int), y_proba_late))

    mlflow.log_artifact("../models/notebook5_state_encoder.pkl")
    mlflow.log_artifact("../models/notebook5_scaler.pkl")
    mlflow.log_artifact("../models/notebook5_rare_states.pkl")
    mlflow.log_artifact("../models/notebook5_feature_names.pkl")

    model_info = mlflow.sklearn.log_model(
        model, artifact_path="model", registered_model_name="olist-late-delivery-model"
    )

    print("Run ID:", mlflow.active_run().info.run_id)
    print("Model URI:", model_info.model_uri)

2026/09/11 08:22:42 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/11 08:22:42 INFO mlflow.store.db.utils: Updating database tables
2026/09/11 08:22:44 INFO mlflow.tracking.fluent: Experiment with name 'olist-late-delivery' does not exist. Creating a new experiment.
2026/09/11 08:22:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Run ID: 9393614ff0754881a797412e2e371ef1
Model URI: models:/m-580ee1e6b7c542b6919005b443937303


Successfully registered model 'olist-late-delivery-model'.
Created version '1' of model 'olist-late-delivery-model'.


In [4]:
from mlflow import MlflowClient

client = MlflowClient()
latest = client.search_model_versions("name='olist-late-delivery-model'")[0]

client.set_registered_model_alias(
    name="olist-late-delivery-model",
    alias="production",
    version=latest.version,
)
print(f"✅ Version {latest.version} tagged with alias 'production'")

✅ Version 1 tagged with alias 'production'
